# 05 - Real vs Synthetic Questionnaire Analysis

This notebook compares real patient questionnaire responses with LLM-generated (synthetic) responses for **all 152 participants**.

**Objectives:**
1. Calculate mean and standard deviation of LLM scores per participant across all repetitions
2. Compute MAE and RMSE between real and synthetic responses (PCS, BPI, TSK)
3. Compare correlation patterns between real and synthetic data
4. Extra: Compare error metrics for expert-evaluated vs non-expert-evaluated subsets

**Data Sources:**
- Real data: `real_patient_pcs`, `real_patient_bpi`, `real_patient_tsk` (N=152)
- Synthetic data: `llm_pcs_results`, `llm_bpi_results`, `llm_tsk_results` (all runs)

**Schema:** `pain_narratives_acm_202512`

## 1. Setup & Configuration

In [ ]:
# Standard library imports
import sys
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Data manipulation
import pandas as pd
import numpy as np
from scipy import stats
from sqlalchemy import text
from sqlmodel import Session

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Add src to path
notebook_dir = Path.cwd()
project_root = notebook_dir.parent
sys.path.insert(0, str(project_root / 'src'))

# Import database manager
from pain_narratives.core.database import DatabaseManager

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.2f}'.format)

# Initialize database manager
db = DatabaseManager()

# Schema
SCHEMA = 'pain_narratives_acm_202512'

print("Libraries imported successfully")
print(f"Working directory: {notebook_dir}")
print(f"Analysis run: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

In [ ]:
# Helper functions for error metrics
def calculate_mae(real, synthetic):
    """Calculate Mean Absolute Error."""
    mask = ~(np.isnan(real) | np.isnan(synthetic))
    if mask.sum() == 0:
        return np.nan
    return np.mean(np.abs(real[mask] - synthetic[mask]))

def calculate_rmse(real, synthetic):
    """Calculate Root Mean Square Error."""
    mask = ~(np.isnan(real) | np.isnan(synthetic))
    if mask.sum() == 0:
        return np.nan
    return np.sqrt(np.mean((real[mask] - synthetic[mask])**2))

def calculate_correlation(real, synthetic):
    """Calculate Pearson correlation coefficient."""
    mask = ~(np.isnan(real) | np.isnan(synthetic))
    if mask.sum() < 3:
        return np.nan, np.nan
    r, p = stats.pearsonr(real[mask], synthetic[mask])
    return r, p

def compute_error_metrics(df, real_col, synth_col):
    """Compute MAE, RMSE, and correlation for a pair of columns."""
    real = df[real_col].values.astype(float)
    synth = df[synth_col].values.astype(float)
    
    mae = calculate_mae(real, synth)
    rmse = calculate_rmse(real, synth)
    r, p = calculate_correlation(real, synth)
    n = (~np.isnan(real) & ~np.isnan(synth)).sum()
    
    return {'n': n, 'mae': mae, 'rmse': rmse, 'r': r, 'p_value': p}

print("Helper functions defined")

## 2. Load Real Patient Questionnaire Data (N=152)

In [ ]:
print("="*80)
print("LOADING REAL PATIENT QUESTIONNAIRE DATA")
print("="*80)

# Load real PCS data
query_real_pcs = f"""
SELECT * FROM {SCHEMA}.real_patient_pcs
ORDER BY participant_id
"""

# Load real BPI data
query_real_bpi = f"""
SELECT * FROM {SCHEMA}.real_patient_bpi
ORDER BY participant_id
"""

# Load real TSK data
query_real_tsk = f"""
SELECT * FROM {SCHEMA}.real_patient_tsk
ORDER BY participant_id
"""

with Session(db.engine) as session:
    df_real_pcs = pd.read_sql(text(query_real_pcs), session.connection())
    df_real_bpi = pd.read_sql(text(query_real_bpi), session.connection())
    df_real_tsk = pd.read_sql(text(query_real_tsk), session.connection())

print(f"\nReal PCS records: {len(df_real_pcs)}")
print(f"Real BPI records: {len(df_real_bpi)}")
print(f"Real TSK records: {len(df_real_tsk)}")

# Quick summary of real data
print(f"\nReal Data Summary:")
print(f"  PCS Total - Mean: {df_real_pcs['pcs_total'].mean():.2f}, SD: {df_real_pcs['pcs_total'].std():.2f}")
print(f"  BPI Total - Mean: {df_real_bpi['bpi_total_mean'].mean():.2f}, SD: {df_real_bpi['bpi_total_mean'].std():.2f}")
print(f"  TSK Total - Mean: {df_real_tsk['tsk_total'].mean():.2f}, SD: {df_real_tsk['tsk_total'].std():.2f}")

## 3. Load LLM (Synthetic) Data - All Runs

In [ ]:
print("="*80)
print("LOADING LLM (SYNTHETIC) DATA - ALL RUNS")
print("="*80)

# Load LLM PCS data (all runs)
query_llm_pcs = f"""
SELECT 
    participant_id, run_number, model,
    pcs_total, pcs_rumination, pcs_magnification, pcs_helplessness,
    pcs_01, pcs_02, pcs_03, pcs_04, pcs_05, pcs_06, pcs_07,
    pcs_08, pcs_09, pcs_10, pcs_11, pcs_12, pcs_13
FROM {SCHEMA}.llm_pcs_results
ORDER BY participant_id, run_number
"""

# Load LLM BPI data (all runs) - includes bpi_total_mean for alignment with real data
query_llm_bpi = f"""
SELECT 
    participant_id, run_number, model,
    bpi_total, bpi_total_mean, bpi_interference_avg, bpi_intensity_avg,
    bpiq11, bpiq12, bpiq13, bpiq14, bpiq15, bpiq16, bpiq17,
    bpiq28, bpiq39, bpiq410, bpiq511
FROM {SCHEMA}.llm_bpi_results
ORDER BY participant_id, run_number
"""

# Load LLM TSK data (all runs)
query_llm_tsk = f"""
SELECT 
    participant_id, run_number, model,
    tsk_total,
    tsk_01, tsk_02, tsk_03, tsk_04, tsk_05, tsk_06,
    tsk_07, tsk_08, tsk_09, tsk_10, tsk_11
FROM {SCHEMA}.llm_tsk_results
ORDER BY participant_id, run_number
"""

with Session(db.engine) as session:
    df_llm_pcs = pd.read_sql(text(query_llm_pcs), session.connection())
    df_llm_bpi = pd.read_sql(text(query_llm_bpi), session.connection())
    df_llm_tsk = pd.read_sql(text(query_llm_tsk), session.connection())

n_runs = df_llm_pcs['run_number'].nunique()

print(f"\nLLM PCS records: {len(df_llm_pcs)} ({n_runs} runs x {len(df_llm_pcs)//n_runs} participants)")
print(f"LLM BPI records: {len(df_llm_bpi)} ({n_runs} runs)")
print(f"LLM TSK records: {len(df_llm_tsk)} ({n_runs} runs)")

print(f"\nRuns available: {sorted(df_llm_pcs['run_number'].unique())}")
print(f"Model used: {df_llm_pcs['model'].unique()[0]}")

## 4. Calculate LLM Mean and SD Across Runs (Per Participant)

In [ ]:
print("="*80)
print("CALCULATING LLM MEAN AND SD PER PARTICIPANT (ACROSS ALL RUNS)")
print("="*80)

# PCS: Calculate mean and SD per participant
pcs_numeric_cols = ['pcs_total', 'pcs_rumination', 'pcs_magnification', 'pcs_helplessness'] + \
                   [f'pcs_{i:02d}' for i in range(1, 14)]

df_llm_pcs_agg = df_llm_pcs.groupby('participant_id')[pcs_numeric_cols].agg(['mean', 'std']).reset_index()
df_llm_pcs_agg.columns = ['participant_id'] + [f'{col}_{stat}' for col, stat in df_llm_pcs_agg.columns[1:]]

print(f"\nPCS aggregated: {len(df_llm_pcs_agg)} participants")
print(f"  LLM PCS Total - Mean: {df_llm_pcs_agg['pcs_total_mean'].mean():.2f}, Avg SD: {df_llm_pcs_agg['pcs_total_std'].mean():.2f}")

# BPI: Calculate mean and SD per participant
# Use bpi_total_mean (0-10 scale) for comparison with real data
bpi_numeric_cols = ['bpi_total', 'bpi_total_mean', 'bpi_interference_avg', 'bpi_intensity_avg'] + \
                   ['bpiq11', 'bpiq12', 'bpiq13', 'bpiq14', 'bpiq15', 'bpiq16', 'bpiq17',
                    'bpiq28', 'bpiq39', 'bpiq410', 'bpiq511']

# Convert to numeric
for col in bpi_numeric_cols:
    df_llm_bpi[col] = pd.to_numeric(df_llm_bpi[col], errors='coerce')

df_llm_bpi_agg = df_llm_bpi.groupby('participant_id')[bpi_numeric_cols].agg(['mean', 'std']).reset_index()
df_llm_bpi_agg.columns = ['participant_id'] + [f'{col}_{stat}' for col, stat in df_llm_bpi_agg.columns[1:]]

print(f"BPI aggregated: {len(df_llm_bpi_agg)} participants")
# Report using bpi_total_mean (0-10 scale, aligned with real data)
print(f"  LLM BPI Total Mean (0-10 scale) - Mean: {df_llm_bpi_agg['bpi_total_mean_mean'].mean():.2f}, Avg SD: {df_llm_bpi_agg['bpi_total_mean_std'].mean():.2f}")

# TSK: Calculate mean and SD per participant
tsk_numeric_cols = ['tsk_total'] + [f'tsk_{i:02d}' for i in range(1, 12)]

# Convert to numeric
for col in tsk_numeric_cols:
    df_llm_tsk[col] = pd.to_numeric(df_llm_tsk[col], errors='coerce')

df_llm_tsk_agg = df_llm_tsk.groupby('participant_id')[tsk_numeric_cols].agg(['mean', 'std']).reset_index()
df_llm_tsk_agg.columns = ['participant_id'] + [f'{col}_{stat}' for col, stat in df_llm_tsk_agg.columns[1:]]

print(f"TSK aggregated: {len(df_llm_tsk_agg)} participants")
print(f"  LLM TSK Total - Mean: {df_llm_tsk_agg['tsk_total_mean'].mean():.2f}, Avg SD: {df_llm_tsk_agg['tsk_total_std'].mean():.2f}")

## 5. Merge Real and Synthetic Data (All 152 Participants)

In [ ]:
print("="*80)
print("MERGING REAL AND SYNTHETIC DATA (ALL 152 PARTICIPANTS)")
print("="*80)

# PCS: Merge real and synthetic (mean across runs)
df_pcs_merged = df_real_pcs.merge(
    df_llm_pcs_agg,
    on='participant_id',
    how='inner',
    suffixes=('_real', '_llm')
)
print(f"\nPCS merged: {len(df_pcs_merged)} participants")

# BPI: Merge real and synthetic (with explicit suffixes to handle name collision for bpi_total_mean)
df_bpi_merged = df_real_bpi.merge(
    df_llm_bpi_agg,
    on='participant_id',
    how='inner',
    suffixes=('_real', '_llm')
)
print(f"BPI merged: {len(df_bpi_merged)} participants")

# TSK: Merge real and synthetic
df_tsk_merged = df_real_tsk.merge(
    df_llm_tsk_agg,
    on='participant_id',
    how='inner',
    suffixes=('_real', '_llm')
)
print(f"TSK merged: {len(df_tsk_merged)} participants")

## 6. MAE, RMSE, and Correlations: Real vs Synthetic (All Participants)

In [ ]:
print("="*80)
print("ERROR METRICS AND CORRELATIONS: REAL VS SYNTHETIC (ALL 152 PARTICIPANTS)")
print("="*80)

# Define score comparisons: (real_col, synth_col, display_name)
pcs_comparisons = [
    ('pcs_total', 'pcs_total_mean', 'PCS Total'),
    ('pcs_rumination', 'pcs_rumination_mean', 'PCS Rumination'),
    ('pcs_magnification', 'pcs_magnification_mean', 'PCS Magnification'),
    ('pcs_helplessness', 'pcs_helplessness_mean', 'PCS Helplessness'),
]

bpi_comparisons = [
    ('bpi_total_mean_real', 'bpi_total_mean_llm', 'BPI Total'),
    ('bpi_interference_mean', 'bpi_interference_avg_mean', 'BPI Interference'),
    ('bpi_intensity_mean', 'bpi_intensity_avg_mean', 'BPI Intensity'),
]

tsk_comparisons = [
    ('tsk_total', 'tsk_total_mean', 'TSK Total'),
]

all_results = []

# PCS
print("\n--- PCS ---")
for real_col, synth_col, name in pcs_comparisons:
    metrics = compute_error_metrics(df_pcs_merged, real_col, synth_col)
    metrics['questionnaire'] = name
    all_results.append(metrics)
    print(f"{name}: MAE={metrics['mae']:.2f}, RMSE={metrics['rmse']:.2f}, r={metrics['r']:.3f} (p={metrics['p_value']:.4f})")

# BPI
print("\n--- BPI ---")
for real_col, synth_col, name in bpi_comparisons:
    metrics = compute_error_metrics(df_bpi_merged, real_col, synth_col)
    metrics['questionnaire'] = name
    all_results.append(metrics)
    print(f"{name}: MAE={metrics['mae']:.2f}, RMSE={metrics['rmse']:.2f}, r={metrics['r']:.3f} (p={metrics['p_value']:.4f})")

# TSK
print("\n--- TSK ---")
for real_col, synth_col, name in tsk_comparisons:
    metrics = compute_error_metrics(df_tsk_merged, real_col, synth_col)
    metrics['questionnaire'] = name
    all_results.append(metrics)
    print(f"{name}: MAE={metrics['mae']:.2f}, RMSE={metrics['rmse']:.2f}, r={metrics['r']:.3f} (p={metrics['p_value']:.4f})")

# Create summary dataframe
df_all_metrics = pd.DataFrame(all_results)
df_all_metrics = df_all_metrics[['questionnaire', 'n', 'mae', 'rmse', 'r', 'p_value']]

In [ ]:
# Display summary table
print("\n" + "="*80)
print("SUMMARY TABLE: ALL 152 PARTICIPANTS")
print("="*80)
print("\nMetrics: Real vs Synthetic (LLM mean across runs)")
print("-" * 80)

df_display = df_all_metrics.copy()
df_display.columns = ['Measure', 'N', 'MAE', 'RMSE', 'r', 'p-value']
print(df_display.to_string(index=False))

## 7. LLM Consistency: Standard Deviation Across Runs

In [ ]:
print("="*80)
print("LLM CONSISTENCY: STANDARD DEVIATION ACROSS RUNS")
print("="*80)
print("\nLower SD indicates more consistent LLM responses across runs.")
print(f"Based on {n_runs} independent runs.\n")

consistency_results = []

# PCS Consistency
print("--- PCS ---")
for col_base in ['pcs_total', 'pcs_rumination', 'pcs_magnification', 'pcs_helplessness']:
    col_std = f'{col_base}_std'
    if col_std in df_llm_pcs_agg.columns:
        mean_std = df_llm_pcs_agg[col_std].mean()
        consistency_results.append({'measure': col_base.upper(), 'mean_sd_across_runs': mean_std})
        print(f"  {col_base}: Mean SD = {mean_std:.2f}")

# BPI Consistency - use bpi_total_mean for aligned comparison
print("\n--- BPI ---")
for col_base in ['bpi_total_mean', 'bpi_interference_avg', 'bpi_intensity_avg']:
    col_std = f'{col_base}_std'
    if col_std in df_llm_bpi_agg.columns:
        mean_std = df_llm_bpi_agg[col_std].mean()
        display_name = 'bpi_total (0-10)' if col_base == 'bpi_total_mean' else col_base
        consistency_results.append({'measure': display_name.upper(), 'mean_sd_across_runs': mean_std})
        print(f"  {display_name}: Mean SD = {mean_std:.2f}")

# TSK Consistency
print("\n--- TSK ---")
for col_base in ['tsk_total']:
    col_std = f'{col_base}_std'
    if col_std in df_llm_tsk_agg.columns:
        mean_std = df_llm_tsk_agg[col_std].mean()
        consistency_results.append({'measure': col_base.upper(), 'mean_sd_across_runs': mean_std})
        print(f"  {col_base}: Mean SD = {mean_std:.2f}")

df_consistency = pd.DataFrame(consistency_results)

## 8. Extra Analysis: Expert-Evaluated vs Non-Expert-Evaluated

In [ ]:
print("="*80)
print("EXTRA: COMPARING EXPERT-EVALUATED VS NON-EXPERT-EVALUATED SUBSETS")
print("="*80)

# Load expert-evaluated participant IDs
query_expert = f"""
SELECT DISTINCT participant_id
FROM {SCHEMA}.expert_dimension_evaluation
WHERE participant_id IS NOT NULL
"""

with Session(db.engine) as session:
    df_expert_ids = pd.read_sql(text(query_expert), session.connection())

expert_ids = set(df_expert_ids['participant_id'].astype(int))

print(f"\nExpert-evaluated participants: {len(expert_ids)}")
print(f"Non-expert-evaluated participants: {152 - len(expert_ids)}")

# Add expert flag to merged dataframes
df_pcs_merged['is_expert'] = df_pcs_merged['participant_id'].isin(expert_ids)
df_bpi_merged['is_expert'] = df_bpi_merged['participant_id'].isin(expert_ids)
df_tsk_merged['is_expert'] = df_tsk_merged['participant_id'].isin(expert_ids)

In [ ]:
# Compare metrics for expert vs non-expert subsets
print("\n" + "-"*80)
print("COMPARISON: EXPERT-EVALUATED vs NON-EXPERT-EVALUATED")
print("-"*80)

comparison_results = []

# PCS
for real_col, synth_col, name in pcs_comparisons:
    # Expert subset
    df_exp = df_pcs_merged[df_pcs_merged['is_expert']]
    metrics_exp = compute_error_metrics(df_exp, real_col, synth_col)
    
    # Non-expert subset
    df_non = df_pcs_merged[~df_pcs_merged['is_expert']]
    metrics_non = compute_error_metrics(df_non, real_col, synth_col)
    
    comparison_results.append({
        'measure': name,
        'expert_n': metrics_exp['n'],
        'expert_mae': metrics_exp['mae'],
        'expert_rmse': metrics_exp['rmse'],
        'expert_r': metrics_exp['r'],
        'non_expert_n': metrics_non['n'],
        'non_expert_mae': metrics_non['mae'],
        'non_expert_rmse': metrics_non['rmse'],
        'non_expert_r': metrics_non['r'],
    })

# BPI
for real_col, synth_col, name in bpi_comparisons:
    df_exp = df_bpi_merged[df_bpi_merged['is_expert']]
    metrics_exp = compute_error_metrics(df_exp, real_col, synth_col)
    
    df_non = df_bpi_merged[~df_bpi_merged['is_expert']]
    metrics_non = compute_error_metrics(df_non, real_col, synth_col)
    
    comparison_results.append({
        'measure': name,
        'expert_n': metrics_exp['n'],
        'expert_mae': metrics_exp['mae'],
        'expert_rmse': metrics_exp['rmse'],
        'expert_r': metrics_exp['r'],
        'non_expert_n': metrics_non['n'],
        'non_expert_mae': metrics_non['mae'],
        'non_expert_rmse': metrics_non['rmse'],
        'non_expert_r': metrics_non['r'],
    })

# TSK
for real_col, synth_col, name in tsk_comparisons:
    df_exp = df_tsk_merged[df_tsk_merged['is_expert']]
    metrics_exp = compute_error_metrics(df_exp, real_col, synth_col)
    
    df_non = df_tsk_merged[~df_tsk_merged['is_expert']]
    metrics_non = compute_error_metrics(df_non, real_col, synth_col)
    
    comparison_results.append({
        'measure': name,
        'expert_n': metrics_exp['n'],
        'expert_mae': metrics_exp['mae'],
        'expert_rmse': metrics_exp['rmse'],
        'expert_r': metrics_exp['r'],
        'non_expert_n': metrics_non['n'],
        'non_expert_mae': metrics_non['mae'],
        'non_expert_rmse': metrics_non['rmse'],
        'non_expert_r': metrics_non['r'],
    })

df_comparison = pd.DataFrame(comparison_results)

# Display comparison table
print("\n")
print(f"{'Measure':<20} | {'Expert (N)':<10} | {'Expert MAE':<10} | {'Expert r':<10} | {'Non-Expert (N)':<14} | {'Non-Expert MAE':<14} | {'Non-Expert r':<12}")
print("-" * 105)
for _, row in df_comparison.iterrows():
    print(f"{row['measure']:<20} | {row['expert_n']:<10} | {row['expert_mae']:<10.2f} | {row['expert_r']:<10.3f} | {row['non_expert_n']:<14} | {row['non_expert_mae']:<14.2f} | {row['non_expert_r']:<12.3f}")

## 9. Publication Tables

In [ ]:
print("="*80)
print("PUBLICATION TABLES")
print("="*80)

# Table 1: Main results (all 152 participants)
print("\n--- Table: Real vs Synthetic Comparison (N=152) ---")
print("\nSynthetic scores = Mean across", n_runs, "independent LLM runs")
print("\n")

df_pub = df_all_metrics.copy()
df_pub['MAE'] = df_pub['mae'].apply(lambda x: f"{x:.2f}")
df_pub['RMSE'] = df_pub['rmse'].apply(lambda x: f"{x:.2f}")
df_pub['r'] = df_pub['r'].apply(lambda x: f"{x:.3f}")
df_pub['p-value'] = df_pub['p_value'].apply(lambda x: '<0.001' if x < 0.001 else f"{x:.3f}")

df_pub_display = df_pub[['questionnaire', 'n', 'MAE', 'RMSE', 'r', 'p-value']]
df_pub_display.columns = ['Questionnaire', 'N', 'MAE', 'RMSE', 'r', 'p-value']
print(df_pub_display.to_string(index=False))

In [ ]:
# Table 2: Expert vs Non-Expert comparison
print("\n--- Table: Expert-Evaluated vs Non-Expert-Evaluated Comparison ---")
print("\n")

df_comp_pub = df_comparison.copy()
df_comp_pub['Expert MAE'] = df_comp_pub['expert_mae'].apply(lambda x: f"{x:.2f}")
df_comp_pub['Expert r'] = df_comp_pub['expert_r'].apply(lambda x: f"{x:.3f}")
df_comp_pub['Non-Expert MAE'] = df_comp_pub['non_expert_mae'].apply(lambda x: f"{x:.2f}")
df_comp_pub['Non-Expert r'] = df_comp_pub['non_expert_r'].apply(lambda x: f"{x:.3f}")

df_comp_display = df_comp_pub[['measure', 'expert_n', 'Expert MAE', 'Expert r', 'non_expert_n', 'Non-Expert MAE', 'Non-Expert r']]
df_comp_display.columns = ['Questionnaire', 'N (Expert)', 'MAE (Expert)', 'r (Expert)', 'N (Non-Expert)', 'MAE (Non-Expert)', 'r (Non-Expert)']
print(df_comp_display.to_string(index=False))

## 10. Export Results

In [ ]:
# Save tables to CSV
output_dir = notebook_dir / 'outputs'
output_dir.mkdir(parents=True, exist_ok=True)

# Export main metrics (all participants)
df_all_metrics.to_csv(output_dir / '05_real_vs_synthetic_all_participants.csv', index=False)

# Export comparison (expert vs non-expert)
df_comparison.to_csv(output_dir / '05_real_vs_synthetic_expert_comparison.csv', index=False)

# Export consistency metrics
df_consistency.to_csv(output_dir / '05_llm_consistency_across_runs.csv', index=False)

print("Tables exported:")
print(f"  - {output_dir / '05_real_vs_synthetic_all_participants.csv'}")
print(f"  - {output_dir / '05_real_vs_synthetic_expert_comparison.csv'}")
print(f"  - {output_dir / '05_llm_consistency_across_runs.csv'}")

## 11. Summary

In [ ]:
print("\n" + "="*80)
print("ANALYSIS COMPLETE")
print("="*80)

print(f"\nData Summary:")
print(f"  - Total participants: 152")
print(f"  - LLM runs analyzed: {n_runs}")
print(f"  - Expert-evaluated subset: {len(expert_ids)} participants")
print(f"  - Non-expert-evaluated subset: {152 - len(expert_ids)} participants")

print(f"\nQuestionnaires Analyzed:")
print(f"  - PCS (Pain Catastrophizing Scale): Total + 3 subscales")
print(f"  - BPI (Brief Pain Inventory): Total + Interference + Intensity")
print(f"  - TSK (Tampa Scale of Kinesiophobia): Total")

print(f"\nKey Metrics:")
print(f"  - MAE: Mean Absolute Error between real and synthetic scores")
print(f"  - RMSE: Root Mean Square Error")
print(f"  - r: Pearson correlation (pattern preservation)")
print(f"  - SD across runs: LLM consistency/reliability")
print(f"  - Cronbach's α: Internal consistency (real vs synthetic)")

print(f"\nNote: Synthetic scores are the MEAN across {n_runs} independent LLM runs")
print(f"      Cronbach's α for synthetic data computed per run, then aggregated")

## 12. Cronbach's Alpha - Internal Consistency Comparison

**Objective**: Compare the internal consistency (Cronbach's α) of real patient questionnaire responses with LLM-generated synthetic responses.

**Approach**:
- Real data: Calculate α once using all 152 participants
- Synthetic data: Calculate α for each run separately, then report mean ± SD across all runs
- Compare real vs synthetic reliability

In [ ]:
def cronbach_alpha(df, item_cols):
    """
    Calculate Cronbach's alpha for internal consistency.
    
    Parameters:
    -----------
    df : DataFrame
        Data containing item columns
    item_cols : list
        List of column names for questionnaire items
        
    Returns:
    --------
    tuple : (alpha, n_complete_cases)
    """
    # Drop rows with missing values
    df_clean = df[item_cols].dropna()
    
    if len(df_clean) < 2:
        return np.nan, len(df_clean)
    
    # Get item data
    item_data = df_clean.values
    
    # Number of items
    n_items = len(item_cols)
    
    if n_items < 2:
        return np.nan, len(df_clean)
    
    # Variance of each item
    item_variances = np.var(item_data, axis=0, ddof=1)
    
    # Variance of total score
    total_variance = np.var(np.sum(item_data, axis=1), ddof=1)
    
    # Cronbach's alpha
    if total_variance == 0:
        return np.nan, len(df_clean)
    
    alpha = (n_items / (n_items - 1)) * (1 - np.sum(item_variances) / total_variance)
    
    return alpha, len(df_clean)

print("✅ Cronbach's alpha function defined")

In [ ]:
print("="*80)
print("CRONBACH'S ALPHA: REAL PATIENT DATA")
print("="*80)

# Define item columns for real data
# PCS: 13 items (pcs_01 to pcs_13)
real_pcs_items = [f'pcs_{i:02d}' for i in range(1, 14)]

# BPI: We'll calculate for interference (6 items) and intensity (4 items) separately
real_bpi_interference_items = ['bpiq11', 'bpiq12', 'bpiq13', 'bpiq15', 'bpiq16', 'bpiq17']
real_bpi_intensity_items = ['bpiq28', 'bpiq39', 'bpiq410', 'bpiq511']

# TSK: 11 items (tsk_01 to tsk_11)
real_tsk_items = [f'tsk_{i:02d}' for i in range(1, 12)]

# Calculate Cronbach's alpha for real data
real_alpha_results = []

# PCS
alpha_pcs_real, n_pcs_real = cronbach_alpha(df_real_pcs, real_pcs_items)
real_alpha_results.append({
    'Questionnaire': 'PCS',
    'Subscale': 'Total (13 items)',
    'Alpha': alpha_pcs_real,
    'N': n_pcs_real
})
print(f"\n📊 PCS (Real):")
print(f"   Cronbach's α = {alpha_pcs_real:.3f} (N = {n_pcs_real})")

# BPI - Interference
alpha_bpi_int_real, n_bpi_int_real = cronbach_alpha(df_real_bpi, real_bpi_interference_items)
real_alpha_results.append({
    'Questionnaire': 'BPI',
    'Subscale': 'Interference (6 items)',
    'Alpha': alpha_bpi_int_real,
    'N': n_bpi_int_real
})
print(f"\n📊 BPI Interference (Real):")
print(f"   Cronbach's α = {alpha_bpi_int_real:.3f} (N = {n_bpi_int_real})")

# BPI - Intensity
alpha_bpi_inten_real, n_bpi_inten_real = cronbach_alpha(df_real_bpi, real_bpi_intensity_items)
real_alpha_results.append({
    'Questionnaire': 'BPI',
    'Subscale': 'Intensity (4 items)',
    'Alpha': alpha_bpi_inten_real,
    'N': n_bpi_inten_real
})
print(f"\n📊 BPI Intensity (Real):")
print(f"   Cronbach's α = {alpha_bpi_inten_real:.3f} (N = {n_bpi_inten_real})")

# TSK
alpha_tsk_real, n_tsk_real = cronbach_alpha(df_real_tsk, real_tsk_items)
real_alpha_results.append({
    'Questionnaire': 'TSK',
    'Subscale': 'Total (11 items)',
    'Alpha': alpha_tsk_real,
    'N': n_tsk_real
})
print(f"\n📊 TSK (Real):")
print(f"   Cronbach's α = {alpha_tsk_real:.3f} (N = {n_tsk_real})")

df_real_alpha = pd.DataFrame(real_alpha_results)

In [ ]:
print("\n" + "="*80)
print("CRONBACH'S ALPHA: SYNTHETIC (LLM) DATA - PER RUN")
print("="*80)
print("\n✅ Using ACM schema tables (already contains item-level data from notebook 04)\n")

# The ACM schema tables already have individual item columns populated
# We just need to query them directly with run_number

# Check what we have in the database
query_check = f"""
SELECT run_number, COUNT(*) as n_records
FROM {SCHEMA}.llm_pcs_results
GROUP BY run_number
ORDER BY run_number
"""

print("📋 Checking available runs in ACM schema:")
with Session(db.engine) as session:
    df_check = pd.read_sql(text(query_check), session.connection())
print(df_check.to_string(index=False))

# Define item columns (matching ACM schema)
synth_pcs_items = [f'pcs_{i:02d}' for i in range(1, 14)]
synth_bpi_interference_items = ['bpiq11', 'bpiq12', 'bpiq13', 'bpiq15', 'bpiq16', 'bpiq17']
synth_bpi_intensity_items = ['bpiq28', 'bpiq39', 'bpiq410', 'bpiq511']
synth_tsk_items = [f'tsk_{i:02d}' for i in range(1, 12)]

# Get unique runs from the loaded data (from section 3)
runs = sorted(df_llm_pcs['run_number'].unique())
synth_alpha_per_run = []

print(f"\n📊 Calculating Cronbach's α for {len(runs)} runs: {[int(r) for r in runs]}\n")

for run in runs:
    print(f"{'─'*80}")
    print(f"Run {int(run)}:")
    print(f"{'─'*80}")
    
    # Filter data for this run from the ALREADY LOADED data (section 3)
    df_pcs_run = df_llm_pcs[df_llm_pcs['run_number'] == run].copy()
    df_bpi_run = df_llm_bpi[df_llm_bpi['run_number'] == run].copy()
    df_tsk_run = df_llm_tsk[df_llm_tsk['run_number'] == run].copy()
    
    print(f"   Data size: PCS={len(df_pcs_run)}, BPI={len(df_bpi_run)}, TSK={len(df_tsk_run)}")
    
    # Check which item columns exist and have data
    pcs_items_available = [col for col in synth_pcs_items if col in df_pcs_run.columns]
    pcs_items_with_data = [col for col in pcs_items_available if df_pcs_run[col].notna().any()]
    
    tsk_items_available = [col for col in synth_tsk_items if col in df_tsk_run.columns]
    tsk_items_with_data = [col for col in tsk_items_available if df_tsk_run[col].notna().any()]
    
    print(f"   Items with data: PCS={len(pcs_items_with_data)}/{len(synth_pcs_items)}, TSK={len(tsk_items_with_data)}/{len(synth_tsk_items)}")
    
    # If items are missing, show which ones
    if len(pcs_items_with_data) < len(synth_pcs_items):
        missing = set(synth_pcs_items) - set(pcs_items_with_data)
        print(f"   ⚠️ PCS missing items: {missing}")
    
    if len(tsk_items_with_data) < len(synth_tsk_items):
        missing = set(synth_tsk_items) - set(tsk_items_with_data)
        print(f"   ⚠️ TSK missing items: {missing}")
    
    # Convert to numeric
    for col in synth_pcs_items:
        if col in df_pcs_run.columns:
            df_pcs_run[col] = pd.to_numeric(df_pcs_run[col], errors='coerce')
    
    for col in synth_bpi_interference_items + synth_bpi_intensity_items:
        if col in df_bpi_run.columns:
            df_bpi_run[col] = pd.to_numeric(df_bpi_run[col], errors='coerce')
    
    for col in synth_tsk_items:
        if col in df_tsk_run.columns:
            df_tsk_run[col] = pd.to_numeric(df_tsk_run[col], errors='coerce')
    
    # PCS - use only items with data
    if len(pcs_items_with_data) >= 2:
        alpha_pcs, n_pcs = cronbach_alpha(df_pcs_run, pcs_items_with_data)
    else:
        alpha_pcs, n_pcs = np.nan, 0
    
    synth_alpha_per_run.append({
        'Run': int(run),
        'Questionnaire': 'PCS',
        'Subscale': f'Total ({len(pcs_items_with_data)} items)',
        'Alpha': alpha_pcs,
        'N': n_pcs
    })
    print(f"   PCS: α = {alpha_pcs:.3f} (N = {n_pcs}, {len(pcs_items_with_data)} items)")
    
    # BPI - Interference
    alpha_bpi_int, n_bpi_int = cronbach_alpha(df_bpi_run, synth_bpi_interference_items)
    synth_alpha_per_run.append({
        'Run': int(run),
        'Questionnaire': 'BPI',
        'Subscale': 'Interference (6 items)',
        'Alpha': alpha_bpi_int,
        'N': n_bpi_int
    })
    print(f"   BPI Interference: α = {alpha_bpi_int:.3f} (N = {n_bpi_int})")
    
    # BPI - Intensity
    alpha_bpi_inten, n_bpi_inten = cronbach_alpha(df_bpi_run, synth_bpi_intensity_items)
    synth_alpha_per_run.append({
        'Run': int(run),
        'Questionnaire': 'BPI',
        'Subscale': 'Intensity (4 items)',
        'Alpha': alpha_bpi_inten,
        'N': n_bpi_inten
    })
    print(f"   BPI Intensity: α = {alpha_bpi_inten:.3f} (N = {n_bpi_inten})")
    
    # TSK - use only items with data
    if len(tsk_items_with_data) >= 2:
        alpha_tsk, n_tsk = cronbach_alpha(df_tsk_run, tsk_items_with_data)
    else:
        alpha_tsk, n_tsk = np.nan, 0
    
    synth_alpha_per_run.append({
        'Run': int(run),
        'Questionnaire': 'TSK',
        'Subscale': f'Total ({len(tsk_items_with_data)} items)',
        'Alpha': alpha_tsk,
        'N': n_tsk
    })
    print(f"   TSK: α = {alpha_tsk:.3f} (N = {n_tsk}, {len(tsk_items_with_data)} items)")

df_synth_alpha_per_run = pd.DataFrame(synth_alpha_per_run)

if len(df_synth_alpha_per_run) > 0:
    print(f"\n✅ Cronbach's α calculated for {len(runs)} runs")
else:
    print("\n❌ No valid alpha calculations")

In [ ]:
print("\n" + "="*80)
print("CRONBACH'S ALPHA: SYNTHETIC DATA AGGREGATED ACROSS RUNS")
print("="*80)

# Aggregate synthetic alpha values across runs (mean and SD)
synth_alpha_agg = df_synth_alpha_per_run.groupby(['Questionnaire', 'Subscale'])['Alpha'].agg(['mean', 'std', 'count']).reset_index()
synth_alpha_agg.columns = ['Questionnaire', 'Subscale', 'Alpha_Mean', 'Alpha_SD', 'N_Runs']

print(f"\n📊 Synthetic Data (Mean across {n_runs} runs):")
print(synth_alpha_agg.to_string(index=False))

# Create comparison table
print("\n" + "="*80)
print("COMPARISON TABLE: REAL VS SYNTHETIC CRONBACH'S ALPHA")
print("="*80)

comparison_alpha = []

for idx, row in df_real_alpha.iterrows():
    q = row['Questionnaire']
    subscale = row['Subscale']
    
    # Find matching synthetic data
    synth_match = synth_alpha_agg[
        (synth_alpha_agg['Questionnaire'] == q) & 
        (synth_alpha_agg['Subscale'] == subscale)
    ]
    
    if len(synth_match) > 0:
        synth_mean = synth_match.iloc[0]['Alpha_Mean']
        synth_sd = synth_match.iloc[0]['Alpha_SD']
        n_runs_data = int(synth_match.iloc[0]['N_Runs'])
        
        comparison_alpha.append({
            'Questionnaire': f"{q} - {subscale}",
            'Real_Alpha': row['Alpha'],
            'Real_N': int(row['N']),
            'Synthetic_Alpha_Mean': synth_mean,
            'Synthetic_Alpha_SD': synth_sd,
            'Synthetic_N_Runs': n_runs_data,
            'Difference': synth_mean - row['Alpha']
        })

df_comparison_alpha = pd.DataFrame(comparison_alpha)

print("\n")
for _, row in df_comparison_alpha.iterrows():
    print(f"\n{row['Questionnaire']}:")
    print(f"   Real:      α = {row['Real_Alpha']:.3f} (N = {row['Real_N']})")
    print(f"   Synthetic: α = {row['Synthetic_Alpha_Mean']:.3f} ± {row['Synthetic_Alpha_SD']:.3f} ({row['Synthetic_N_Runs']} runs)")
    print(f"   Difference: Δ = {row['Difference']:+.3f}")

In [ ]:
print("\n" + "="*80)
print("PUBLICATION TABLE: CRONBACH'S ALPHA COMPARISON")
print("="*80)

# Format for publication
df_alpha_pub = df_comparison_alpha.copy()

# Format columns
df_alpha_pub['Real Sample'] = df_alpha_pub.apply(
    lambda row: f"{row['Real_Alpha']:.3f} (n={row['Real_N']})",
    axis=1
)

df_alpha_pub['Synthetic Sample'] = df_alpha_pub.apply(
    lambda row: f"{row['Synthetic_Alpha_Mean']:.3f} ± {row['Synthetic_Alpha_SD']:.3f} ({row['Synthetic_N_Runs']} runs)",
    axis=1
)

df_alpha_pub['Δ'] = df_alpha_pub['Difference'].apply(lambda x: f"{x:+.3f}")

df_alpha_pub_final = df_alpha_pub[['Questionnaire', 'Real Sample', 'Synthetic Sample', 'Δ']]

print("\n")
print(df_alpha_pub_final.to_string(index=False))

print("\n" + "─"*80)
print("📖 Interpretation Guide:")
print("─"*80)
print("   Cronbach's α ≥ 0.70: Acceptable internal consistency")
print("   Cronbach's α ≥ 0.80: Good internal consistency")
print("   Cronbach's α ≥ 0.90: Excellent internal consistency")
print("")
print("   |Δ| < 0.05: Very similar reliability")
print("   |Δ| < 0.10: Similar reliability")
print("   |Δ| ≥ 0.10: Notable difference")

# Export results
df_comparison_alpha.to_csv(output_dir / '05_cronbach_alpha_comparison.csv', index=False)
df_synth_alpha_per_run.to_csv(output_dir / '05_cronbach_alpha_per_run.csv', index=False)

print(f"\n✅ Tables exported:")
print(f"  - {output_dir / '05_cronbach_alpha_comparison.csv'}")
print(f"  - {output_dir / '05_cronbach_alpha_per_run.csv'}")

## 13. Statistical Significance Testing

**Objective**: Perform rigorous statistical comparisons between real and synthetic data to understand:
1. Whether means differ significantly (paired t-tests)
2. Consistency across repetitions (ANOVA)
3. Effect sizes (Cohen's d)
4. Per-run comparisons

**Note on BPI Scale**: Real BPI stores **means** (0-10 scale averaged across items), while synthetic BPI stores **sums** (total of all item values). This causes a ~10x ratio but both are valid representations.

In [ ]:
# Statistical helper functions
def cohen_d(group1, group2):
    """Calculate Cohen's d effect size."""
    n1, n2 = len(group1), len(group2)
    var1, var2 = np.var(group1, ddof=1), np.var(group2, ddof=1)
    pooled_std = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
    return (np.mean(group1) - np.mean(group2)) / pooled_std if pooled_std > 0 else np.nan

def interpret_cohens_d(d):
    """Interpret Cohen's d effect size."""
    abs_d = abs(d)
    if abs_d < 0.2:
        return "negligible"
    elif abs_d < 0.5:
        return "small"
    elif abs_d < 0.8:
        return "medium"
    else:
        return "large"

print("Statistical helper functions defined")

In [ ]:
print("="*80)
print("13.1 PAIRED T-TESTS: REAL VS SYNTHETIC (MEAN ACROSS RUNS)")
print("="*80)
print("\nComparing real patient scores with synthetic scores (averaged across all runs)")
print("H0: Mean difference = 0 (no significant difference)")
print("H1: Mean difference ≠ 0 (significant difference)")
print("\n")

paired_test_results = []

# PCS Total
real_pcs = df_pcs_merged['pcs_total'].values
synth_pcs = df_pcs_merged['pcs_total_mean'].values
t_stat, p_val = stats.ttest_rel(real_pcs, synth_pcs)
effect_size = cohen_d(real_pcs, synth_pcs)

paired_test_results.append({
    'Questionnaire': 'PCS Total',
    'Real_Mean': real_pcs.mean(),
    'Real_SD': real_pcs.std(),
    'Synthetic_Mean': synth_pcs.mean(),
    'Synthetic_SD': synth_pcs.std(),
    'Mean_Diff': real_pcs.mean() - synth_pcs.mean(),
    't_statistic': t_stat,
    'p_value': p_val,
    'cohens_d': effect_size,
    'effect_interpretation': interpret_cohens_d(effect_size)
})

print(f"PCS Total:")
print(f"  Real:      M={real_pcs.mean():.2f}, SD={real_pcs.std():.2f}")
print(f"  Synthetic: M={synth_pcs.mean():.2f}, SD={synth_pcs.std():.2f}")
print(f"  t({len(real_pcs)-1})={t_stat:.3f}, p={p_val:.4f}, d={effect_size:.3f} ({interpret_cohens_d(effect_size)})")
print(f"  {'✓ Significant' if p_val < 0.05 else '✗ Not significant'} at α=0.05\n")

# BPI Total (now aligned on 0-10 scale)
real_bpi_total = df_bpi_merged['bpi_total_mean_real'].values
synth_bpi_total = df_bpi_merged['bpi_total_mean_mean'].values
t_stat, p_val = stats.ttest_rel(real_bpi_total, synth_bpi_total)
effect_size = cohen_d(real_bpi_total, synth_bpi_total)

paired_test_results.append({
    'Questionnaire': 'BPI Total',
    'Real_Mean': real_bpi_total.mean(),
    'Real_SD': real_bpi_total.std(),
    'Synthetic_Mean': synth_bpi_total.mean(),
    'Synthetic_SD': synth_bpi_total.std(),
    'Mean_Diff': real_bpi_total.mean() - synth_bpi_total.mean(),
    't_statistic': t_stat,
    'p_value': p_val,
    'cohens_d': effect_size,
    'effect_interpretation': interpret_cohens_d(effect_size)
})

print(f"BPI Total (0-10 scale):")
print(f"  Real:      M={real_bpi_total.mean():.2f}, SD={real_bpi_total.std():.2f}")
print(f"  Synthetic: M={synth_bpi_total.mean():.2f}, SD={synth_bpi_total.std():.2f}")
print(f"  t({len(real_bpi_total)-1})={t_stat:.3f}, p={p_val:.4f}, d={effect_size:.3f} ({interpret_cohens_d(effect_size)})")
print(f"  {'✓ Significant' if p_val < 0.05 else '✗ Not significant'} at α=0.05\n")

# BPI Interference (comparable scales)
real_bpi_int = df_bpi_merged['bpi_interference_mean'].values
synth_bpi_int = df_bpi_merged['bpi_interference_avg_mean'].values
t_stat, p_val = stats.ttest_rel(real_bpi_int, synth_bpi_int)
effect_size = cohen_d(real_bpi_int, synth_bpi_int)

paired_test_results.append({
    'Questionnaire': 'BPI Interference',
    'Real_Mean': real_bpi_int.mean(),
    'Real_SD': real_bpi_int.std(),
    'Synthetic_Mean': synth_bpi_int.mean(),
    'Synthetic_SD': synth_bpi_int.std(),
    'Mean_Diff': real_bpi_int.mean() - synth_bpi_int.mean(),
    't_statistic': t_stat,
    'p_value': p_val,
    'cohens_d': effect_size,
    'effect_interpretation': interpret_cohens_d(effect_size)
})

print(f"BPI Interference:")
print(f"  Real:      M={real_bpi_int.mean():.2f}, SD={real_bpi_int.std():.2f}")
print(f"  Synthetic: M={synth_bpi_int.mean():.2f}, SD={synth_bpi_int.std():.2f}")
print(f"  t({len(real_bpi_int)-1})={t_stat:.3f}, p={p_val:.4f}, d={effect_size:.3f} ({interpret_cohens_d(effect_size)})")
print(f"  {'✓ Significant' if p_val < 0.05 else '✗ Not significant'} at α=0.05\n")

# BPI Intensity (comparable scales)
real_bpi_inten = df_bpi_merged['bpi_intensity_mean'].values
synth_bpi_inten = df_bpi_merged['bpi_intensity_avg_mean'].values
t_stat, p_val = stats.ttest_rel(real_bpi_inten, synth_bpi_inten)
effect_size = cohen_d(real_bpi_inten, synth_bpi_inten)

paired_test_results.append({
    'Questionnaire': 'BPI Intensity',
    'Real_Mean': real_bpi_inten.mean(),
    'Real_SD': real_bpi_inten.std(),
    'Synthetic_Mean': synth_bpi_inten.mean(),
    'Synthetic_SD': synth_bpi_inten.std(),
    'Mean_Diff': real_bpi_inten.mean() - synth_bpi_inten.mean(),
    't_statistic': t_stat,
    'p_value': p_val,
    'cohens_d': effect_size,
    'effect_interpretation': interpret_cohens_d(effect_size)
})

print(f"BPI Intensity:")
print(f"  Real:      M={real_bpi_inten.mean():.2f}, SD={real_bpi_inten.std():.2f}")
print(f"  Synthetic: M={synth_bpi_inten.mean():.2f}, SD={synth_bpi_inten.std():.2f}")
print(f"  t({len(real_bpi_inten)-1})={t_stat:.3f}, p={p_val:.4f}, d={effect_size:.3f} ({interpret_cohens_d(effect_size)})")
print(f"  {'✓ Significant' if p_val < 0.05 else '✗ Not significant'} at α=0.05\n")

# TSK Total
real_tsk = df_tsk_merged['tsk_total'].values
synth_tsk = df_tsk_merged['tsk_total_mean'].values
t_stat, p_val = stats.ttest_rel(real_tsk, synth_tsk)
effect_size = cohen_d(real_tsk, synth_tsk)

paired_test_results.append({
    'Questionnaire': 'TSK Total',
    'Real_Mean': real_tsk.mean(),
    'Real_SD': real_tsk.std(),
    'Synthetic_Mean': synth_tsk.mean(),
    'Synthetic_SD': synth_tsk.std(),
    'Mean_Diff': real_tsk.mean() - synth_tsk.mean(),
    't_statistic': t_stat,
    'p_value': p_val,
    'cohens_d': effect_size,
    'effect_interpretation': interpret_cohens_d(effect_size)
})

print(f"TSK Total:")
print(f"  Real:      M={real_tsk.mean():.2f}, SD={real_tsk.std():.2f}")
print(f"  Synthetic: M={synth_tsk.mean():.2f}, SD={synth_tsk.std():.2f}")
print(f"  t({len(real_tsk)-1})={t_stat:.3f}, p={p_val:.4f}, d={effect_size:.3f} ({interpret_cohens_d(effect_size)})")
print(f"  {'✓ Significant' if p_val < 0.05 else '✗ Not significant'} at α=0.05\n")

df_paired_tests = pd.DataFrame(paired_test_results)

print("-"*80)
print("Summary:")
print("-"*80)
sig_count = (df_paired_tests['p_value'] < 0.05).sum()
print(f"{sig_count}/{len(df_paired_tests)} comparisons show significant differences (p < 0.05)")

In [ ]:
print("\n" + "="*80)
print("13.2 ANOVA: CONSISTENCY ACROSS RUNS (Within Synthetic Data)")
print("="*80)
print("\nTesting if synthetic scores differ significantly across runs 1, 2, and 3")
print("H0: All runs have the same mean (consistent across repetitions)")
print("H1: At least one run differs significantly")
print("\n")

anova_results = []

# PCS Total - compare across runs
pcs_run1 = df_llm_pcs[df_llm_pcs['run_number'] == 1]['pcs_total'].values
pcs_run2 = df_llm_pcs[df_llm_pcs['run_number'] == 2]['pcs_total'].values
pcs_run3 = df_llm_pcs[df_llm_pcs['run_number'] == 3]['pcs_total'].values
f_stat, p_val = stats.f_oneway(pcs_run1, pcs_run2, pcs_run3)

anova_results.append({
    'Questionnaire': 'PCS Total',
    'Run1_Mean': pcs_run1.mean(),
    'Run2_Mean': pcs_run2.mean(),
    'Run3_Mean': pcs_run3.mean(),
    'Overall_Mean': np.concatenate([pcs_run1, pcs_run2, pcs_run3]).mean(),
    'F_statistic': f_stat,
    'p_value': p_val,
    'Consistent': 'Yes' if p_val >= 0.05 else 'No'
})

print(f"PCS Total:")
print(f"  Run 1: M={pcs_run1.mean():.2f}, SD={pcs_run1.std():.2f}")
print(f"  Run 2: M={pcs_run2.mean():.2f}, SD={pcs_run2.std():.2f}")
print(f"  Run 3: M={pcs_run3.mean():.2f}, SD={pcs_run3.std():.2f}")
print(f"  F({2}, {len(pcs_run1)*3-3})={f_stat:.3f}, p={p_val:.4f}")
print(f"  {'✓ Consistent' if p_val >= 0.05 else '✗ Inconsistent'} across runs (α=0.05)\n")

# BPI Interference
bpi_int_run1 = df_llm_bpi[df_llm_bpi['run_number'] == 1]['bpi_interference_avg'].values
bpi_int_run2 = df_llm_bpi[df_llm_bpi['run_number'] == 2]['bpi_interference_avg'].values
bpi_int_run3 = df_llm_bpi[df_llm_bpi['run_number'] == 3]['bpi_interference_avg'].values
f_stat, p_val = stats.f_oneway(bpi_int_run1, bpi_int_run2, bpi_int_run3)

anova_results.append({
    'Questionnaire': 'BPI Interference',
    'Run1_Mean': bpi_int_run1.mean(),
    'Run2_Mean': bpi_int_run2.mean(),
    'Run3_Mean': bpi_int_run3.mean(),
    'Overall_Mean': np.concatenate([bpi_int_run1, bpi_int_run2, bpi_int_run3]).mean(),
    'F_statistic': f_stat,
    'p_value': p_val,
    'Consistent': 'Yes' if p_val >= 0.05 else 'No'
})

print(f"BPI Interference:")
print(f"  Run 1: M={bpi_int_run1.mean():.2f}, SD={bpi_int_run1.std():.2f}")
print(f"  Run 2: M={bpi_int_run2.mean():.2f}, SD={bpi_int_run2.std():.2f}")
print(f"  Run 3: M={bpi_int_run3.mean():.2f}, SD={bpi_int_run3.std():.2f}")
print(f"  F({2}, {len(bpi_int_run1)*3-3})={f_stat:.3f}, p={p_val:.4f}")
print(f"  {'✓ Consistent' if p_val >= 0.05 else '✗ Inconsistent'} across runs (α=0.05)\n")

# BPI Intensity
bpi_inten_run1 = df_llm_bpi[df_llm_bpi['run_number'] == 1]['bpi_intensity_avg'].values
bpi_inten_run2 = df_llm_bpi[df_llm_bpi['run_number'] == 2]['bpi_intensity_avg'].values
bpi_inten_run3 = df_llm_bpi[df_llm_bpi['run_number'] == 3]['bpi_intensity_avg'].values
f_stat, p_val = stats.f_oneway(bpi_inten_run1, bpi_inten_run2, bpi_inten_run3)

anova_results.append({
    'Questionnaire': 'BPI Intensity',
    'Run1_Mean': bpi_inten_run1.mean(),
    'Run2_Mean': bpi_inten_run2.mean(),
    'Run3_Mean': bpi_inten_run3.mean(),
    'Overall_Mean': np.concatenate([bpi_inten_run1, bpi_inten_run2, bpi_inten_run3]).mean(),
    'F_statistic': f_stat,
    'p_value': p_val,
    'Consistent': 'Yes' if p_val >= 0.05 else 'No'
})

print(f"BPI Intensity:")
print(f"  Run 1: M={bpi_inten_run1.mean():.2f}, SD={bpi_inten_run1.std():.2f}")
print(f"  Run 2: M={bpi_inten_run2.mean():.2f}, SD={bpi_inten_run2.std():.2f}")
print(f"  Run 3: M={bpi_inten_run3.mean():.2f}, SD={bpi_inten_run3.std():.2f}")
print(f"  F({2}, {len(bpi_inten_run1)*3-3})={f_stat:.3f}, p={p_val:.4f}")
print(f"  {'✓ Consistent' if p_val >= 0.05 else '✗ Inconsistent'} across runs (α=0.05)\n")

# TSK Total
tsk_run1 = df_llm_tsk[df_llm_tsk['run_number'] == 1]['tsk_total'].values
tsk_run2 = df_llm_tsk[df_llm_tsk['run_number'] == 2]['tsk_total'].values
tsk_run3 = df_llm_tsk[df_llm_tsk['run_number'] == 3]['tsk_total'].values
f_stat, p_val = stats.f_oneway(tsk_run1, tsk_run2, tsk_run3)

anova_results.append({
    'Questionnaire': 'TSK Total',
    'Run1_Mean': tsk_run1.mean(),
    'Run2_Mean': tsk_run2.mean(),
    'Run3_Mean': tsk_run3.mean(),
    'Overall_Mean': np.concatenate([tsk_run1, tsk_run2, tsk_run3]).mean(),
    'F_statistic': f_stat,
    'p_value': p_val,
    'Consistent': 'Yes' if p_val >= 0.05 else 'No'
})

print(f"TSK Total:")
print(f"  Run 1: M={tsk_run1.mean():.2f}, SD={tsk_run1.std():.2f}")
print(f"  Run 2: M={tsk_run2.mean():.2f}, SD={tsk_run2.std():.2f}")
print(f"  Run 3: M={tsk_run3.mean():.2f}, SD={tsk_run3.std():.2f}")
print(f"  F({2}, {len(tsk_run1)*3-3})={f_stat:.3f}, p={p_val:.4f}")
print(f"  {'✓ Consistent' if p_val >= 0.05 else '✗ Inconsistent'} across runs (α=0.05)\n")

df_anova = pd.DataFrame(anova_results)

print("-"*80)
print("Summary:")
print("-"*80)
consistent_count = (df_anova['Consistent'] == 'Yes').sum()
print(f"{consistent_count}/{len(df_anova)} questionnaires show consistent means across runs (p ≥ 0.05)")

In [ ]:
print("\n" + "="*80)
print("13.3 BPI SCALE ALIGNMENT VERIFICATION")
print("="*80)
print("\nVerifying Real and Synthetic BPI data use the same scale (0-10)")
print("\n")

# Check a few examples from real data
print("Real BPI Sample (first 5 participants):")
print("-"*80)
sample_real = df_real_bpi[['participant_id', 'bpi_total_mean', 'bpi_interference_mean', 'bpi_intensity_mean']].head()
print(sample_real.to_string(index=False))

print("\n")
print("Synthetic BPI Sample (first 5 participants, run 1):")
print("-"*80)
sample_synth = df_llm_bpi[df_llm_bpi['run_number'] == 1][['participant_id', 'bpi_total_mean', 'bpi_interference_avg', 'bpi_intensity_avg']].head()
print(sample_synth.to_string(index=False))

print("\n" + "="*80)
print("SCALE CONFIRMATION")
print("="*80)
print("\nBoth Real and Synthetic BPI data now use the same scale:")
print("  - bpi_total_mean: MEAN of all item responses (0-10 scale)")
print("  - bpi_interference: Mean of interference items (0-10 scale)")
print("  - bpi_intensity: Mean of intensity items (0-10 scale)")
print("\nThis alignment enables direct statistical comparison without scale conversion.")

In [ ]:
print("\n" + "="*80)
print("13.4 STATISTICAL SUMMARY TABLE")
print("="*80)
print("\n")

# Create comprehensive summary table
summary_stats = []

for idx, row in df_paired_tests.iterrows():
    q = row['Questionnaire']
    
    # Find matching ANOVA result
    anova_match = df_anova[df_anova['Questionnaire'] == q]
    
    if len(anova_match) > 0:
        summary_stats.append({
            'Questionnaire': q,
            'Real_Mean': f"{row['Real_Mean']:.2f}",
            'Real_SD': f"{row['Real_SD']:.2f}",
            'Synth_Mean': f"{row['Synthetic_Mean']:.2f}",
            'Synth_SD': f"{row['Synthetic_SD']:.2f}",
            'Mean_Diff': f"{row['Mean_Diff']:.2f}",
            't_test_p': f"{row['p_value']:.4f}" if row['p_value'] >= 0.001 else '<0.001',
            'Cohens_d': f"{row['cohens_d']:.3f}",
            'Effect': row['effect_interpretation'],
            'ANOVA_p': f"{anova_match.iloc[0]['p_value']:.4f}" if anova_match.iloc[0]['p_value'] >= 0.001 else '<0.001',
            'Runs_Consistent': anova_match.iloc[0]['Consistent']
        })

df_summary = pd.DataFrame(summary_stats)

print("Statistical Test Results:")
print("-"*120)
print(f"{'Questionnaire':<18} | {'Real M(SD)':<12} | {'Synth M(SD)':<12} | {'Δ':<8} | {'t-test p':<9} | {'d':<7} | {'Effect':<10} | {'ANOVA p':<9} | {'Consistent'}")
print("-"*120)

for _, row in df_summary.iterrows():
    real_str = f"{row['Real_Mean']}({row['Real_SD']})"
    synth_str = f"{row['Synth_Mean']}({row['Synth_SD']})"
    print(f"{row['Questionnaire']:<18} | {real_str:<12} | {synth_str:<12} | {row['Mean_Diff']:<8} | {row['t_test_p']:<9} | {row['Cohens_d']:<7} | {row['Effect']:<10} | {row['ANOVA_p']:<9} | {row['Runs_Consistent']}")

print("\n" + "─"*80)
print("Interpretation:")
print("─"*80)
print("  - Δ: Mean difference (Real - Synthetic)")
print("  - t-test p: Paired t-test p-value (Real vs Synthetic mean)")
print("  - d: Cohen's d effect size")
print("  - ANOVA p: One-way ANOVA p-value (across runs 1-3)")
print("  - Consistent: Whether runs have similar means (Yes if ANOVA p ≥ 0.05)")

In [ ]:
print("\n" + "="*80)
print("13.5 EXPORT STATISTICAL RESULTS")
print("="*80)

# Export paired t-tests
df_paired_tests.to_csv(output_dir / '05_statistical_paired_ttests.csv', index=False)
print(f"✓ Exported: {output_dir / '05_statistical_paired_ttests.csv'}")

# Export ANOVA results
df_anova.to_csv(output_dir / '05_statistical_anova_across_runs.csv', index=False)
print(f"✓ Exported: {output_dir / '05_statistical_anova_across_runs.csv'}")

# Export summary
df_summary.to_csv(output_dir / '05_statistical_summary.csv', index=False)
print(f"✓ Exported: {output_dir / '05_statistical_summary.csv'}")

print("\n" + "="*80)
print("KEY FINDINGS")
print("="*80)

print("\n1. Real vs Synthetic Comparison (Paired t-tests):")
sig_diffs = df_paired_tests[df_paired_tests['p_value'] < 0.05]
if len(sig_diffs) > 0:
    print(f"   - {len(sig_diffs)}/{len(df_paired_tests)} questionnaires show significant mean differences")
    for _, row in sig_diffs.iterrows():
        print(f"     • {row['Questionnaire']}: Δ={row['Mean_Diff']:.2f}, d={row['cohens_d']:.3f} ({row['effect_interpretation']})")
else:
    print("   - No significant differences found")

print("\n2. Consistency Across Runs (ANOVA):")
consistent = df_anova[df_anova['Consistent'] == 'Yes']
print(f"   - {len(consistent)}/{len(df_anova)} questionnaires show consistent means across runs")
if len(consistent) == len(df_anova):
    print("   - ✓ All questionnaires demonstrate excellent reproducibility")
else:
    inconsistent = df_anova[df_anova['Consistent'] == 'No']
    for _, row in inconsistent.iterrows():
        print(f"     • {row['Questionnaire']}: F={row['F_statistic']:.3f}, p={row['p_value']:.4f}")

print("\n3. BPI Scale Note:")
print("   - Real BPI uses MEAN (0-10 scale), Synthetic uses SUM")
print("   - This causes ~10x ratio but both are valid measurements")
print("   - Use BPI Interference and Intensity subscales for direct comparison")

print("\n" + "="*80)

## 14. Population-Level vs Individual-Level Agreement

**Key Insight**: Statistical significance tests (t-tests) and error metrics (MAE, RMSE) measure fundamentally different things:

| Level | Metrics | Question Answered |
|-------|---------|-------------------|
| **Population** | t-test, Cohen's d, mean/SD comparison | "Do the two groups look similar overall?" |
| **Individual** | MAE, RMSE, Pearson r | "Does participant X's synthetic score match their real score?" |

These can diverge: populations can be statistically similar while individual predictions have high error.


In [ ]:
print("="*80)
print("14.1 AGREEMENT LEVELS: POPULATION VS INDIVIDUAL")
print("="*80)

# Build comprehensive comparison table
agreement_data = []

# Helper function to interpret correlation
def interpret_r(r):
    r = abs(r)
    if r >= 0.7: return "Strong"
    elif r >= 0.5: return "Moderate"
    elif r >= 0.3: return "Weak"
    else: return "Very Weak"

# Helper function to interpret Cohen's d
def interpret_d(d):
    d = abs(d)
    if d >= 0.8: return "Large"
    elif d >= 0.5: return "Medium"
    elif d >= 0.2: return "Small"
    else: return "Negligible"

# Get data from previous analyses (df_paired_tests and df_all_metrics should exist)
questionnaires = ['PCS Total', 'BPI Total', 'BPI Interference', 'BPI Intensity', 'TSK Total']

for q in questionnaires:
    row = {'Questionnaire': q}
    
    # Population-level metrics (from paired t-tests)
    ttest_row = df_paired_tests[df_paired_tests['Questionnaire'] == q]
    if len(ttest_row) > 0:
        ttest_row = ttest_row.iloc[0]
        row['Real_Mean'] = ttest_row['Real_Mean']
        row['Real_SD'] = ttest_row['Real_SD']
        row['Synth_Mean'] = ttest_row['Synthetic_Mean']
        row['Synth_SD'] = ttest_row['Synthetic_SD']
        row['Mean_Diff'] = ttest_row['Mean_Diff']
        row['t_stat'] = ttest_row['t_statistic']
        row['p_value'] = ttest_row['p_value']
        row['Cohens_d'] = ttest_row['cohens_d']
        row['Pop_Sig'] = 'Yes' if ttest_row['p_value'] < 0.05 else 'No'
    
    # Individual-level metrics (from error metrics)
    # Map questionnaire names
    q_map = {
        'PCS Total': 'PCS Total',
        'BPI Total': 'BPI Total',
        'BPI Interference': 'BPI Interference',
        'BPI Intensity': 'BPI Intensity',
        'TSK Total': 'TSK Total'
    }
    metrics_row = df_all_metrics[df_all_metrics['questionnaire'] == q_map.get(q, q)]
    if len(metrics_row) > 0:
        metrics_row = metrics_row.iloc[0]
        row['MAE'] = metrics_row['mae']
        row['RMSE'] = metrics_row['rmse']
        row['Pearson_r'] = metrics_row['r']
        row['r_pvalue'] = metrics_row['p_value']
    
    agreement_data.append(row)

df_agreement = pd.DataFrame(agreement_data)

print("\n" + "-"*80)
print("POPULATION-LEVEL AGREEMENT (Do groups look similar overall?)")
print("-"*80)
print(f"{'Questionnaire':<18} {'Real':<14} {'Synthetic':<14} {'Diff':<8} {'t':<8} {'p':<8} {'d':<8} {'Sig?':<6}")
print("-"*80)

for _, row in df_agreement.iterrows():
    real_str = f"{row.get('Real_Mean', 0):.2f}({row.get('Real_SD', 0):.2f})"
    synth_str = f"{row.get('Synth_Mean', 0):.2f}({row.get('Synth_SD', 0):.2f})"
    print(f"{row['Questionnaire']:<18} {real_str:<14} {synth_str:<14} {row.get('Mean_Diff', 0):>+7.2f} {row.get('t_stat', 0):>7.2f} {row.get('p_value', 1):>7.4f} {row.get('Cohens_d', 0):>7.3f} {row.get('Pop_Sig', 'N/A'):<6}")

print("\n" + "-"*80)
print("INDIVIDUAL-LEVEL AGREEMENT (Can we predict specific participant scores?)")
print("-"*80)
print(f"{'Questionnaire':<18} {'MAE':<10} {'RMSE':<10} {'r':<10} {'r Strength':<12}")
print("-"*80)

for _, row in df_agreement.iterrows():
    r_val = row.get('Pearson_r', 0)
    print(f"{row['Questionnaire']:<18} {row.get('MAE', 0):<10.2f} {row.get('RMSE', 0):<10.2f} {r_val:<10.3f} {interpret_r(r_val):<12}")


In [ ]:
print("\n" + "="*80)
print("14.2 INTERPRETATION FRAMEWORK")
print("="*80)

print("""
UNDERSTANDING THE RESULTS:
--------------------------

1. POPULATION-LEVEL (t-tests, Cohen's d):
   - Non-significant p-values (p > 0.05) indicate groups are statistically similar
   - Small Cohen's d (< 0.2) indicates negligible practical difference
   
   → If both conditions met: LLM generates data with similar DISTRIBUTIONAL PROPERTIES
     to real patient data (same means, similar spread)

2. INDIVIDUAL-LEVEL (MAE, RMSE, Pearson r):
   - MAE/RMSE: Average error in predicting individual scores
   - Pearson r: How well individual rank-ordering is preserved
   
   → High MAE + Moderate r: LLM captures general patterns but not exact values
   → This is EXPECTED behavior - narratives don't contain all information needed
     for precise score prediction

3. THE KEY DISTINCTION:
   
   ┌─────────────────────────────────────────────────────────────────────┐
   │  DISTRIBUTIONAL VALIDITY  ≠  INDIVIDUAL PREDICTION ACCURACY        │
   │                                                                     │
   │  The LLM can generate a POPULATION of synthetic patients that      │
   │  statistically resembles real chronic pain patients, WITHOUT       │
   │  being able to precisely predict what SPECIFIC individuals scored. │
   └─────────────────────────────────────────────────────────────────────┘

WHY THIS MAKES SCIENTIFIC SENSE:
--------------------------------
A narrative captures SOME but not ALL information about a patient's psychological state.

Example: A narrative might convey "severe pain with catastrophizing tendencies"
  → LLM correctly generates HIGH PCS score
  → But exact score (38 vs 42) depends on nuances not in the narrative

This is analogous to:
- Knowing someone is "tall" lets you generate plausible heights
- But you can't predict their exact height from that description alone
""")


In [ ]:
print("\n" + "="*80)
print("14.3 SUGGESTED TEXT FOR ACM PAPER")
print("="*80)

# Calculate summary statistics for paper
n_pop_similar = (df_agreement['Pop_Sig'] == 'No').sum()
n_total = len(df_agreement)
avg_mae_pcs = df_agreement[df_agreement['Questionnaire'] == 'PCS Total']['MAE'].values[0] if len(df_agreement) > 0 else 0
avg_r = df_agreement['Pearson_r'].mean()

print("""
RESULTS SECTION TEXT (adapt as needed):
---------------------------------------

"We evaluated the validity of LLM-generated questionnaire responses at two levels:
population-level (distributional similarity) and individual-level (prediction accuracy).

POPULATION-LEVEL VALIDITY:
Paired t-tests comparing real patient scores with LLM-generated synthetic scores
(averaged across {n_runs} independent runs) revealed {n_similar}/{n_total} questionnaire 
measures showed no statistically significant differences (p > 0.05). Effect sizes 
(Cohen's d) were uniformly small to negligible, indicating the synthetic data 
preserves the distributional properties of real patient responses.

INDIVIDUAL-LEVEL ACCURACY:
While population distributions were well-matched, individual-level prediction 
showed moderate accuracy. Mean Absolute Errors ranged from X.XX to Y.YY points,
with Pearson correlations between real and synthetic scores ranging from r = 0.XX 
to r = 0.YY (all p < 0.001). This indicates the LLM preserves relative rank-ordering
of patients while showing expected variation in absolute score prediction.

INTERPRETATION:
These findings suggest LLMs can generate synthetic patient populations that are
statistically representative of real chronic pain patients for research purposes.
However, the moderate individual-level accuracy indicates that narrative text alone
does not contain sufficient information to precisely predict specific questionnaire 
responses. This is consistent with the theoretical understanding that standardized
questionnaires capture psychological constructs not fully expressed in free-text
narratives."
""".format(n_runs=n_runs, n_similar=n_pop_similar, n_total=n_total))

print("\n" + "-"*80)
print("KEY STATISTICS TO REPORT:")
print("-"*80)

for _, row in df_agreement.iterrows():
    q = row['Questionnaire']
    print(f"\n{q}:")
    print(f"  Population: Real M={row.get('Real_Mean', 0):.2f} (SD={row.get('Real_SD', 0):.2f}), "
          f"Synthetic M={row.get('Synth_Mean', 0):.2f} (SD={row.get('Synth_SD', 0):.2f})")
    print(f"  t-test: t={row.get('t_stat', 0):.2f}, p={row.get('p_value', 1):.4f}, d={row.get('Cohens_d', 0):.3f}")
    print(f"  Individual: MAE={row.get('MAE', 0):.2f}, RMSE={row.get('RMSE', 0):.2f}, r={row.get('Pearson_r', 0):.3f}")


In [ ]:
print("\n" + "="*80)
print("14.4 EXPORT AGREEMENT ANALYSIS")
print("="*80)

# Export the comprehensive agreement table
df_agreement.to_csv(output_dir / '05_agreement_levels_comparison.csv', index=False)
print(f"\nExported to: {output_dir / '05_agreement_levels_comparison.csv'}")

# Create a formatted summary for easy copy-paste
summary_text = []
summary_text.append("POPULATION-LEVEL VS INDIVIDUAL-LEVEL AGREEMENT SUMMARY")
summary_text.append("=" * 60)
summary_text.append("")

for _, row in df_agreement.iterrows():
    summary_text.append(f"{row['Questionnaire']}:")
    summary_text.append(f"  Population: {'Similar' if row.get('Pop_Sig') == 'No' else 'Different'} "
                       f"(p={row.get('p_value', 1):.4f}, d={row.get('Cohens_d', 0):.3f})")
    summary_text.append(f"  Individual: r={row.get('Pearson_r', 0):.3f} ({interpret_r(row.get('Pearson_r', 0))}), "
                       f"MAE={row.get('MAE', 0):.2f}")
    summary_text.append("")

summary_path = output_dir / '05_agreement_summary.txt'
with open(summary_path, 'w') as f:
    f.write("\n".join(summary_text))
print(f"Exported summary to: {summary_path}")

print("\n" + "="*80)
print("SECTION 14 COMPLETE")
print("="*80)
